# 02 — TF-IDF & Cosine Similarity

How classical NLP turns text into **numbers**. This notebook builds TF-IDF
*by hand first*, then shows scikit-learn producing the same thing.

**Concepts:** bag of words → TF → IDF → TF-IDF → vector space → cosine similarity

Production use: `backend/app/nlp/intent.py` (the baseline classifier) and the
feature space every classical model in Phase 3 trains on.
Docs: `docs/nlp/03_classification.md`

In [ ]:
import math, re, sys
from pathlib import Path

cwd = Path.cwd()
backend_dir = cwd.parent / "backend" if cwd.name == "notebooks" else cwd / "backend"
sys.path.insert(0, str(backend_dir))

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
print("imports OK")

## 1. Bag of Words — the first vectorization

A document becomes a **count profile over the vocabulary**. Word order is
thrown away; frequency is kept. This lossy step is what makes linear algebra
on text possible.

In [ ]:
corpus = [
    "i forgot my password",
    "i cannot remember my password",
    "what is the weather today",
    "will it rain tomorrow",
]
vec = CountVectorizer()
X = vec.fit_transform(corpus)
pd.DataFrame(X.toarray(), columns=vec.get_feature_names_out(), index=[f"doc{i}" for i in range(4)])

## 2. TF — Term Frequency

```
TF(t, d) = count(t in d) / |d|
```

Normalize by document length so "the word appears 5 times" means different
things in a 5-word and a 500-word document. sklearn uses raw counts by
default; we will use `sublinear_tf = 1 + log(count)` later.

In [ ]:
def tf(counts):
    total = counts.sum()
    return counts / total if total else counts

counts = X.toarray()[0].astype(float)
print("raw   :", dict(zip(vec.get_feature_names_out(), counts)))
print("tf    :", dict(zip(vec.get_feature_names_out(), tf(counts))))

## 3. IDF — Inverse Document Frequency

```
IDF(t) = log( N / df(t) )
```

- `df(t)` = number of documents containing `t`
- If `t` is in ALL documents: `log(N/N) = 0` → carries **zero** topical signal
- If `t` is in 1 of N: `log(N)` → very informative

The logarithm damps the ratio (a word 100x rarer is not 100x "10x more
important"). TF × IDF combines *local importance* × *global rarity*.

In [ ]:
N = len(corpus)

def idf(term):
    df = sum(1 for doc in corpus if re.search(rf"\b{re.escape(term)}\b", doc))
    return math.log(N / df) if df else 0.0

for term in ["password", "weather", "forgot"]:
    df = sum(1 for doc in corpus if re.search(rf"\b{re.escape(term)}\b", doc))
    print(f"{term:10} df={df}  IDF=log({N}/{df})={idf(term):.3f}")

## 4. TF-IDF vector, by hand

`TF-IDF(t,d) = TF(t,d) × IDF(t)` — one number per (word, document) pair.

In [ ]:
vocab = vec.get_feature_names_out()
matrix = []
for i, doc in enumerate(corpus):
    counts = X.toarray()[i].astype(float)
    row = tf(counts) * np.array([idf(t) for t in vocab])
    matrix.append(row)

df = pd.DataFrame(matrix, columns=vocab, index=[f"doc{i}" for i in range(4)])
df.round(3)

### Same idea in scikit-learn

`TfidfVectorizer` = CountVectorizer + TF rule + IDF + **L2 normalization**
(unit length vectors, which makes cosine a plain dot product).
Our production settings: `sublinear_tf=True`, `ngram_range=(1,2)`,
`stop_words=None` (see the docs for why stopwords *hurt* intent detection).

In [ ]:
prod = TfidfVectorizer(sublinear_tf=True, ngram_range=(1, 2), stop_words=None)
X_t = prod.fit_transform(corpus)
pd.DataFrame(X_t.toarray(), columns=prod.get_feature_names_out(),
             index=[f"doc{i}" for i in range(4)]).round(3)

## 5. Cosine similarity — comparing two texts

```
cosine(A, B) = (A · B) / (‖A‖ ‖B‖)
```

The **angle** between vectors: `1` = same direction (same words, same
proportions), `0` = orthogonal (nothing in common). Because TF-IDF vectors are
L2-normalized, cosine = dot product.

In [ ]:
def cosine(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

a, b = matrix[0], matrix[1]   # both about passwords
d = matrix[2]                 # about weather
print(f"doc0 vs doc1 (both passwords): {cosine(a, b):.3f}")
print(f"doc0 vs doc2 (password/weather): {cosine(a, d):.3f}")
print("sklearn check              :", cosine_similarity([a], [b])[0][0].round(3))

## 6. Retrieval demo — find the closest pattern

This is *exactly* how the production baseline classifier predicts an intent:
query → TF-IDF → cosine against every training pattern → argmax.

In [ ]:
patterns = ["i forgot my password", "reset my login credentials",
            "what is the weather today", "will it rain tomorrow"]
labels   = ["password_help", "password_help", "weather", "weather"]

search = TfidfVectorizer(sublinear_tf=True, ngram_range=(1, 2))
P = search.fit_transform(patterns + ["i cannot remember my login password"])
q = P[-1]
scores = cosine_similarity(q, P[:-1]).ravel()
for pat, lab, s in sorted(zip(patterns, labels, scores), key=lambda x: -x[2]):
    print(f"{s:.3f}  {lab:15} {pat}")

## 7. Limitations — why Phase 5 exists

TF-IDF cannot see meaning, only shared *strings*:

| query | top TF-IDF match | problem |
|---|---|---|
| "i can't remember my login credentials" | weather? | zero shared keywords with "forgot password" |
| "fluffy kittens" vs "cute cats" | no match | synonyms don't overlap |
| "not good" vs "good" | near-identical | negation invisible |

Sentence embeddings (Phase 5) fix this by placing *meaning* near meaning in
vector space — same cosine machinery, semantic vectors instead of word counts.

## 8. Connection to production

```python
from app.nlp.intent import classify_intent
classify_intent("hey, what's up?")   # -> IntentResult(label='greeting', ...)
```

The service loads `artifacts/intent/*.joblib` (trained once by
`python -m app.nlp.train_intent_model`), never retrains at startup.

In [ ]:
from app.nlp.intent import classify_intent
for msg in ["hey, what's up?", "I can't remember my login password", "purple monkey dishwasher"]:
    r = classify_intent(msg)
    print(f"{msg!r:50} -> {r.label:16} {r.confidence:.2f} ({r.method})")